In [155]:
import sys
import subprocess
import os

# Pure Python environment mapping bypassing terminal shell limits
try:
    import tensorflow as tf
    import sklearn
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "tensorflow", "scikit-learn", "pandas", "numpy", "matplotlib", "seaborn"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense
from tensorflow.keras.layers import TextVectorization
from sklearn.model_selection import train_test_split

# Force strict reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Synthetic e-commerce data footprint generation
n_reviews = 1500
positive_phrases = ["Excellent product", "Highly recommend this", "Amazing quality", "Fast shipping, great", "Perfect fit and comfortable"]
negative_phrases = ["Arrived broken and unusable", "Terrible quality, do not buy", "Very unhappy with this purchase", "Waste of money", "Cheap material and broke instantly"]

simulated_data = {
    'review_text': [np.random.choice(positive_phrases if i % 2 == 0 else negative_phrases) + f" {idx}" for idx, i in enumerate(range(n_reviews))],
    'rating': np.random.choice([1, 2, 3, 4, 5], size=n_reviews, p=[0.2, 0.15, 0.1, 0.15, 0.4])
}

df_raw = pd.DataFrame(simulated_data)
df_raw.iloc[np.random.choice(n_reviews, 15, replace=False), 0] = np.nan
df_raw.to_csv('ecommerce_reviews.csv', index=False)

# Clean, filter, and binarize
df = pd.read_csv('ecommerce_reviews.csv')
df.dropna(subset=['review_text'], inplace=True)
df = df[df['rating'] != 3].copy()
df['sentiment'] = df['rating'].apply(lambda r: 1 if r >= 4 else 0)

X_train, X_test, y_train, y_test = train_test_split(
    df['review_text'].values, df['sentiment'].values, test_size=0.20, random_state=42, stratify=df['sentiment'].values
)
print(f"🧹 Setup Complete. Training pool size: {len(X_train)}")

🧹 Setup Complete. Training pool size: 1072


In [156]:
VOCAB_SIZE = 10000
MAX_SEQUENCE_LENGTH = 100
EMBEDDING_DIM = 32

vectorizer_layer = TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_SEQUENCE_LENGTH,
    standardize='lower_and_strip_punctuation'
)
vectorizer_layer.adapt(X_train)
print("🎯 TextVectorization vocabulary dictionary adapted successfully.")

🎯 TextVectorization vocabulary dictionary adapted successfully.


In [157]:
# Armored structural layout fully compatible with Keras 2 and Keras 3 engines
model = Sequential([
    vectorizer_layer,
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM),
    GlobalAveragePooling1D(),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['binary_accuracy']
)

# Execute the 10-epoch tracking loop
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

# Render convergence metrics to headless background files
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.savefig('sentiment_nn_curves.png', dpi=150)
plt.close()

Epoch 1/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - binary_accuracy: 0.6129 - loss: 0.6706 - val_binary_accuracy: 0.6157 - val_loss: 0.6662
Epoch 2/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - binary_accuracy: 0.6166 - loss: 0.6663 - val_binary_accuracy: 0.6157 - val_loss: 0.6661
Epoch 3/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - binary_accuracy: 0.6166 - loss: 0.6660 - val_binary_accuracy: 0.6157 - val_loss: 0.6661
Epoch 4/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - binary_accuracy: 0.6166 - loss: 0.6658 - val_binary_accuracy: 0.6157 - val_loss: 0.6661
Epoch 5/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - binary_accuracy: 0.6166 - loss: 0.6657 - val_binary_accuracy: 0.6157 - val_loss: 0.6661
Epoch 6/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - binary_accuracy: 0.6166 - loss: 0.6655 - val_binary_accuracy: 0.6157 - val_loss: 0.6661
Epoch 7/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - binary_accuracy: 0.6166 - loss: 0.6654 - val_binary_accuracy: 0.6157 - val_loss: 0.6661
Epoch 8/10
34/34 ━

In [158]:
print("\n--- 🔎 MANDATORY STRING VALIDATION TEST ---")
mandatory_test_string = "The product arrived broken and I am very unhappy"

# Direct vector array inference evaluation
raw_prediction = model.predict([mandatory_test_string], verbose=0)[0][0]
print(f"Target Input String : '{mandatory_test_string}'")
print(f"Model Confidence Score Output: {raw_prediction:.6f}")


--- 🔎 MANDATORY STRING VALIDATION TEST ---


ValueError: Unrecognized data type: x=['The product arrived broken and I am very unhappy'] (of type <class 'list'>)

In [159]:
print("\n--- 🔎 MANDATORY STRING VALIDATION TEST ---")
mandatory_test_string = "The product arrived broken and I am very unhappy"

# Direct vector array inference evaluation
raw_prediction = model.predict([mandatory_test_string], verbose=0)[0][0]
print(f"Target Input String : '{mandatory_test_string}'")
print(f"Model Confidence Score Output: {raw_prediction:.6f}")


--- 🔎 MANDATORY STRING VALIDATION TEST ---


ValueError: Unrecognized data type: x=['The product arrived broken and I am very unhappy'] (of type <class 'list'>)